In [1]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# =====================================================================
# 1. MOCK DATA: Simulem les 1068 Seccions Censals de BCN
# =====================================================================
np.random.seed(42)
n_seccions = 1068

# Generem dades falses però realistes per a les 3 variables estrella
renda = np.random.normal(35000, 15000, n_seccions) # Renda mitjana
edat_edificis = np.random.randint(1900, 2020, n_seccions) # Any de construcció
avis_sols = np.random.randint(5, 120, n_seccions) # Nombre de llars amb avis sols

# Simulem l'Índex CIMNE (Target Y): Renda baixa + Pis vell + Avis = Risc Alt
risc_base = (60000 - renda) * 0.001 + (2020 - edat_edificis) * 0.2 + (avis_sols * 0.15)
soroll = np.random.normal(0, 4, n_seccions)
index_cimne_mock = np.clip(risc_base + soroll, 0, 100) # De 0 a 100

# Creem el DataFrame d'entrada
df = pd.DataFrame({
    'Renda_Llar': renda,
    'Any_Construccio': edat_edificis,
    'Avis_Sols': avis_sols,
    'Index_CIMNE': index_cimne_mock
})

# =====================================================================
# 2. PREPROCESSAMENT (Molt important per al TFG)
# =====================================================================
X = df[['Renda_Llar', 'Any_Construccio', 'Avis_Sols']]
y = df['Index_CIMNE']

# Divisió de dades (85% Train / 15% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# Escalem les dades (Vital per a la Xarxa Neuronal MLP)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =====================================================================
# 3. ENTRENAMENT DELS 3 MODELS
# =====================================================================
# A. Regressió Lineal (Baseline)
model_lr = LinearRegression()
model_lr.fit(X_train_scaled, y_train)
pred_lr = model_lr.predict(X_test_scaled)

# B. Random Forest (Estat de l'art en dades tabulars)
model_rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
model_rf.fit(X_train_scaled, y_train)
pred_rf = model_rf.predict(X_test_scaled)

# C. Perceptró Multicapa - MLP (Deep Learning)
# Arquitectura: 2 capes ocultes de 64 i 32 neurones
model_mlp = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=2000, random_state=42)
model_mlp.fit(X_train_scaled, y_train)
pred_mlp = model_mlp.predict(X_test_scaled)

# =====================================================================
# 4. AVALUACIÓ I RESULTATS
# =====================================================================
print("📊 RESULTATS DEL PIPELINE (TEST SET - 15% DADES NO VISTES) 📊\n")

def avaluar_model(nom, y_real, y_pred):
    mae = mean_absolute_error(y_real, y_pred)
    r2 = r2_score(y_real, y_pred)
    print(f"--- {nom} ---")
    print(f"MAE (Error mig): {mae:.2f} punts")
    print(f"R2 Score:        {r2:.3f}\n")

avaluar_model("Regressió Lineal", y_test, pred_lr)
avaluar_model("Random Forest", y_test, pred_rf)
avaluar_model("Xarxa Neuronal (MLP)", y_test, pred_mlp)

# Extracció de regles (L'avantatge del Random Forest)
print("🔍 IMPORTÀNCIA DE LES VARIABLES (Només disponible al Random Forest):")
importancies = model_rf.feature_importances_
for nom_var, imp in zip(X.columns, importancies):
    print(f" - {nom_var}: {imp*100:.1f}%")

📊 RESULTATS DEL PIPELINE (TEST SET - 15% DADES NO VISTES) 📊

--- Regressió Lineal ---
MAE (Error mig): 2.96 punts
R2 Score:        0.952

--- Random Forest ---
MAE (Error mig): 3.13 punts
R2 Score:        0.941

--- Xarxa Neuronal (MLP) ---
MAE (Error mig): 2.89 punts
R2 Score:        0.953

🔍 IMPORTÀNCIA DE LES VARIABLES (Només disponible al Random Forest):
 - Renda_Llar: 73.2%
 - Any_Construccio: 18.0%
 - Avis_Sols: 8.8%


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# =====================================================================
# 1. CÀRREGA DEL DATASET REAL (Les teves 1068 files)
# =====================================================================
df = pd.read_csv('../data/processed/03_master_dataset_final.csv')

# Seleccionem NOMÉS les 3 variables estrella del TFG
columnes_x = [
    'renda_disponible_Import_Euros', 
    'edificacions_any_Any_Mitja_Ponderat',
    'pad_dom_Llars_1_Avi_Sol'
]

# Netegem possibles NaNs que hagin quedat a aquestes 3 columnes
df_clean = df.dropna(subset=columnes_x).copy()
X = df_clean[columnes_x]

# =====================================================================
# 2. CREACIÓ DE LA VARIABLE RESPOSTA (Y) SIMULADA
# =====================================================================
np.random.seed(42)

# Lògica: Renda baixa + Pis vell + Avis = Risc Alt
y_simulada = (
    (60000 - X['renda_disponible_Import_Euros']) * 0.001 + 
    (2025 - X['edificacions_any_Any_Mitja_Ponderat']) * 0.2 + 
    (X['pad_dom_Llars_1_Avi_Sol'] * 0.15)
)

# Limitem l'Índex entre 0 i 100 i hi posem soroll
y_simulada = np.clip(y_simulada + np.random.normal(0, 4, len(X)), 0, 100)

# =====================================================================
# 3. PIPELINE DE MACHINE LEARNING
# =====================================================================
X_train, X_test, y_train, y_test = train_test_split(X, y_simulada, test_size=0.15, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Executem els 3 models
models = {
    "Regressió Lineal": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Xarxa Neuronal (MLP)": MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=2000, random_state=42)
}

print(f"🚀 Llançant prova amb {len(df_clean)} seccions reals de Barcelona...\n")

for nom, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    print(f"--- {nom} ---")
    print(f"MAE: {mean_absolute_error(y_test, preds):.2f} | R2: {r2_score(y_test, preds):.3f}\n")

# Importància de variables (Només del Random Forest)
rf_model = models["Random Forest"]
print("🔍 IMPORTÀNCIA DE LES VARIABLES (Només disponible al Random Forest):")
for nom_var, imp in zip(columnes_x, rf_model.feature_importances_):
    print(f" - {nom_var}: {imp*100:.1f}%")

🚀 Llançant prova amb 1068 seccions reals de Barcelona...

--- Regressió Lineal ---
MAE: 4.24 | R2: 0.409

--- Random Forest ---
MAE: 1.27 | R2: 0.874

--- Xarxa Neuronal (MLP) ---
MAE: 1.37 | R2: 0.890

🔍 IMPORTÀNCIA DE LES VARIABLES (Només disponible al Random Forest):
 - renda_disponible_Import_Euros: 28.4%
 - edificacions_any_Any_Mitja_Ponderat: 3.6%
 - pad_dom_Llars_1_Avi_Sol: 68.0%
